In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import date
from sklearn.metrics import root_mean_squared_error

from aquacrop import AquaCrop, Crop, Soil, Weather, InitialConditions
from aquacrop.templates import manuloa_co2_records

from aquacrop_slovenia import config
from aquacrop_slovenia.prepare_weather import load_station_weather
from aquacrop_slovenia.parameter_defaults_jablje import jablje_soil_layers, jablje_curve_number, jablje_readily_evaporable_water, jablje_maize_params
from aquacrop_slovenia.management_defaults import optimal_management
from aquacrop_slovenia.intial_conditions_defaults import intial_cond_params
from aquacrop_slovenia.plots import plot_yield_timeseries_comparison, plot_yield_scatter
from aquacrop_slovenia.yield_data import get_yield, get_yield_for_comparison

In [ ]:
# Set up working directory for outputs
output_dir = config.MODELS_DIR / "testing"

plant and end dates are set to be the same every year - TODO TEMPORARY!!!

We exclude 2017 since we dont have ET data for most of the year

In [ ]:
#year = 1993
simulation_periods = [
    {
        "start_date": date(year, 1, 1),
        "end_date": date(year, 10, 10), # TODO temporary
        "planting_date": date(year, 5, 3), # TODO temporary
        "is_seeding_year": True,
    }
#]
for year in range(1993, 2024) if year != 2017]

In [ ]:
jablje_temperatures, jablje_eto, jablje_rain = load_station_weather(8) # Jablje uses ARSO meteo station 8 data

In [ ]:
jablje_weather = Weather(
    location="Jablje",
    temperatures=jablje_temperatures,
    eto_values=jablje_eto,
    rainfall_values=jablje_rain,
    record_type=1,
    first_day=1,
    first_month=1,
    first_year=1993,
    co2_records=manuloa_co2_records
)

In [ ]:
jablje_maize = Crop(
    name="Jablje Maize",
    description="Jablje maize uncalibrated",
    params=jablje_maize_params
)

In [ ]:
jablje_soil = Soil(
    name="Jablje Soil",
    description="Jablje silt loam soil uncalibrated",
    soil_layers=jablje_soil_layers,
    curve_number=jablje_curve_number,
    readily_evaporable_water=jablje_readily_evaporable_water
)

In [ ]:
default_initial_conditions = InitialConditions(
    name="DefaultInitialConditions",
    description="Default initial conditions with AquaCrop calculated defaults",
    params=intial_cond_params
)

In [ ]:
# Create AquaCrop simulation
simulation = AquaCrop(
    simulation_periods=simulation_periods,
    crop=jablje_maize,
    soil=jablje_soil,
    management=optimal_management,
    initial_conditions=default_initial_conditions,
    climate=jablje_weather,
    working_dir=output_dir,
    need_daily_output=True,
    need_seasonal_output=True,
    need_harvest_output=False,
    need_evaluation_output=False
)

In [ ]:
# Run the simulation
results = simulation.run()

In [ ]:
results["season"]

In [ ]:
results["season"].plot(x="Year1", y="Y(dry)")

In [ ]:
get_yield("jablje", "A", "N0")

In [ ]:
plot_yield_timeseries_comparison(results["season"][["Year1", "Y(dry)"]], get_yield("jablje", "A", "N0"))

In [ ]:
plot_yield_scatter(results["season"][["Year1", "Y(dry)"]], get_yield("jablje", "A", "N0"))

In [ ]:
root_mean_squared_error(get_yield_for_comparison("jablje", "A", "N0"), results["season"][["Year1", "Y(dry)"]])

In [ ]:
results["day"]

In [ ]:
results["day"].plot(y="Biomass")

In [ ]:
results["day"].plot(y="CC")

In [ ]:
results["day"].plot(y="Tr")

In [ ]:
results["day"].plot(y="Y(dry)")

In [ ]:
from aquacrop_slovenia.diagnostics import compute_annual_gdd

base_temp = maize_params['base_temp']
upper_temp = maize_params['upper_temp']

gdd_by_year = compute_annual_gdd(8, base_temp, upper_temp)
gdd_by_year

In [ ]:
from aquacrop_slovenia.diagnostics import plot_gdd_and_yield

plot_gdd_and_yield(gdd_by_year, results["season"], title="Annual GDD and Maize Dry Yield — Jablje")